## Simple GenAI app using Langchain

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")   ## --> LangSmith Tracking
os.environ['LANGCHAIN_TRACING_V2'] = "true"                        ## ---> LangSmith Tracing
os.environ['LANGCHAIN_PROJECT'] = os.getenv("LANGCHAIN_PROJECT")

In [6]:
## Data Ingestion
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://docs.langchain.com/langsmith/home")
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, t

In [7]:
## Dividing docs into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
split = splitter.split_documents(docs)

In [8]:
split

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, t

In [16]:
## Embedding Models by Google
import google.generativeai as genai

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

for m in genai.list_models():
    if "embed" in m.name.lower():
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview


In [13]:
## Converting Text into Vectors
from langchain_community.embeddings import OllamaEmbeddings
embedding_ollama = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

## OR

from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
embedding_google = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [14]:
from langchain_community.vectorstores import FAISS

vector_ollama = FAISS.from_documents(documents=split, embedding=embedding_ollama)
vector_google = FAISS.from_documents(documents=split, embedding=embedding_google)

In [19]:
query = "How to create an Account"
result_ollama = vector_ollama.similarity_search(query=query)
result_ollama[0].page_content

'Edit this page on GitHub or file an issue.Connect these docs to Claude, VSCode, and more via MCP for real-time answers.Was this page helpful?YesNoCreate an account and API keyNext⌘IDocs by LangChain home pagegithubxlinkedinyoutubeResourcesForumChangelogLangChain AcademyContact SalesCompanyHomeTrust CenterCareersBloggithubxlinkedinyoutube'

In [20]:
query = "How to create an API key"
result_google = vector_google.similarity_search(query=query)
result_google[0].page_content

'\u200bGet started\nCreate an accountSign up at smith.langchain.com (no credit card required).\nYou can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.\nCopy the key and save it securely.Choose your integrationLangSmith works with many frameworks and providers. Browse available integrations to connect your stack.\nOnce your account and API key are ready, choose a quickstart to begin building with LangSmith:\nObservabilityGain visibility into every step your application takes to debug faster and improve reliability.Start tracingEvaluationMeasure and track quality over time to ensure your AI applications are consistent and trustworthy.Evaluate your appDeploymentDeploy your agents as Agent Servers, ready to scale in production.Deploy your agents\n\u200bMore ways to build'

### Retrieval and Document Chain

In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0.7
)

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)

prompt = ChatPromptTemplate.from_template(
    """
            Answer the following question based only on the provided context:
<context>
{context}
<context>
    """
)

document_chain = create_stuff_documents_chain(llm=llm , prompt=prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n            Answer the following question based only on the provided context:\n<context>\n{context}\n<context>\n    '), additional_kwargs={})])
| ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True,

In [25]:
from langchain_classic.chains import create_retrieval_chain
retriever_google = vector_google.as_retriever()
retriever_ollama = vector_ollama.as_retriever()

retrieve_chain_google = create_retrieval_chain(retriever_google, document_chain)

In [27]:
retrieve_chain_google

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001DAD3E074D0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n            Answer the following question based only on the provided context:\n<context>\n{context}\n<context>\n    '), additional_kwar

In [29]:
## get the Response from LLM
response_google = retrieve_chain_google.invoke({
    "input" : "How to create an API key"
})
response_google['answer']

'To create an API key, go to your Settings page, then navigate to API Keys, and click on "Create API Key". After creating it, copy the key and save it securely.'

In [30]:
response_google['context']

[Document(id='89162edd-96ee-455b-888e-efea81e13829', metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='\u200bGet started\nCreate an accountSign up at smith.langchain.com (no credit card required).\nYou can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.\nCopy the key and save it securely.Choose your integrationLangSmith works with many frameworks and providers. Browse available integrations to connect your stack.\nOnce your account and API key are ready, choose a quickstart to begin building with LangSmith:\nObservabilityGain visibility into every step your application takes to debug faster and improve reliability.Start tracingEvaluationMeasure and track quality over time to ensure your AI applications are consistent and trustworthy.Evaluate your appDeploymentDeploy your agents as Agent Servers, ready to scale in production.Deploy yo